# Error Analysis - Nepali News Classification

This notebook provides comprehensive error analysis for the trained Nepali news classification model, identifying patterns in misclassifications and areas for improvement.

## Objectives:
1. Load model and test predictions
2. Analyze confusion patterns between classes
3. Identify most confused class pairs
4. Analyze error patterns by text characteristics (length, confidence)
5. Examine specific misclassified samples
6. Generate visualizations of error patterns
7. Provide insights for model improvement


In [ ]:
# Import required libraries
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from tqdm import tqdm
from collections import defaultdict, Counter
from sklearn.metrics import confusion_matrix

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

# Import custom modules
try:
    from model import create_model, NepaliNewsClassifier
    from transformers import XLMRobertaTokenizer
    print("✓ Custom modules imported successfully")
except ImportError as e:
    print(f"✗ Error importing custom modules: {e}")
    raise

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")
if torch.cuda.is_available():
    print(f"✓ CUDA Device: {torch.cuda.get_device_name(0)}")
print(f"✓ PyTorch version: {torch.__version__}")


## 1. Load Model and Test Predictions


In [ ]:
# Load model configuration
models_dir = project_root / 'models'
summary_path = models_dir / 'training_summary.json'

if summary_path.exists():
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    MODEL_NAME = summary['model_name']
    NUM_CLASSES = summary['num_classes']
    print(f"✓ Model: {MODEL_NAME}, Classes: {NUM_CLASSES}")
else:
    MODEL_NAME = 'xlm-roberta-base'
    NUM_CLASSES = 20
    print("⚠ Using default configuration")

MAX_LENGTH = 512
DROPOUT_RATE = 0.3

# Load model checkpoint
checkpoint_path = models_dir / 'best_model.pt'
if not checkpoint_path.exists():
    checkpoint_path = models_dir / 'final_model.pt'

checkpoint = torch.load(checkpoint_path, map_location=device)
if 'model_name' in checkpoint:
    MODEL_NAME = checkpoint['model_name']
if 'num_classes' in checkpoint:
    NUM_CLASSES = checkpoint['num_classes']
if 'dropout_rate' in checkpoint:
    DROPOUT_RATE = checkpoint['dropout_rate']
if 'max_length' in checkpoint:
    MAX_LENGTH = checkpoint['max_length']

print(f"✓ Loaded checkpoint from: {checkpoint_path.name}")

# Initialize tokenizer and model
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_NAME)
model = create_model(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES,
    dropout_rate=DROPOUT_RATE,
    device=device
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"✓ Model loaded and ready")


In [ ]:
# Load test dataset
data_dir = project_root / 'data' / 'splits'
test_df = pd.read_csv(data_dir / 'test.csv')

# Load label mappings
label_mappings_path = project_root / 'data' / 'processed' / 'label_mappings.json'
with open(label_mappings_path, 'r', encoding='utf-8') as f:
    label_mappings = json.load(f)

id_to_label = {int(k): v for k, v in label_mappings['id_to_label'].items()}
label_to_id = {v: int(k) for k, v in label_mappings['id_to_label'].items()}

print(f"✓ Test samples: {len(test_df):,}")
print(f"✓ Categories: {NUM_CLASSES}")

# Create dataset class
class NepaliNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create test dataset and loader
test_dataset = NepaliNewsDataset(
    texts=test_df['text'].tolist(),
    labels=test_df['category_id'].tolist(),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)
print(f"✓ Test data loader created")


In [ ]:
# Get predictions on test set
print("Generating predictions on test set...")
all_predictions = []
all_labels = []
all_probs = []
all_texts = test_df['text'].tolist()

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=-1)
        predictions = torch.argmax(logits, dim=-1).cpu().numpy()
        
        all_predictions.extend(predictions)
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

print("✓ Predictions generated")

# Create results dataframe
results_df = pd.DataFrame({
    'text': all_texts,
    'true_label': all_labels,
    'predicted_label': all_predictions,
    'true_class': [id_to_label[int(l)] for l in all_labels],
    'predicted_class': [id_to_label[int(p)] for p in all_predictions],
    'confidence': [np.max(prob) for prob in all_probs],
    'is_correct': [l == p for l, p in zip(all_labels, all_predictions)]
})

# Add text characteristics
results_df['text_length'] = results_df['text'].str.len()
results_df['word_count'] = results_df['text'].str.split().str.len()

print(f"\n✓ Results dataframe created")
print(f"Total samples: {len(results_df):,}")
print(f"Correct predictions: {results_df['is_correct'].sum():,} ({results_df['is_correct'].mean()*100:.2f}%)")
print(f"Misclassified: {(~results_df['is_correct']).sum():,} ({(~results_df['is_correct']).mean()*100:.2f}%)")


## 2. Confusion Matrix Analysis


In [ ]:
# Generate confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

# Find most confused class pairs
confusion_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'true_class': id_to_label[i],
                'predicted_class': id_to_label[j],
                'count': int(cm[i, j]),
                'true_id': i,
                'predicted_id': j
            })

# Sort by count
confusion_pairs = sorted(confusion_pairs, key=lambda x: x['count'], reverse=True)

print("=" * 80)
print("TOP 20 MOST CONFUSED CLASS PAIRS")
print("=" * 80)
print(f"{'True Class':<25} {'Predicted Class':<25} {'Count':<10} {'% of Errors':<15}")
print("-" * 80)
total_errors = sum(pair['count'] for pair in confusion_pairs)
for i, pair in enumerate(confusion_pairs[:20], 1):
    pct = (pair['count'] / total_errors * 100) if total_errors > 0 else 0
    print(f"{pair['true_class']:<25} {pair['predicted_class']:<25} {pair['count']:<10} {pct:>6.2f}%")
print("=" * 80)


In [ ]:
# Visualize top confused pairs
top_pairs = confusion_pairs[:15]
fig, ax = plt.subplots(figsize=(14, 8))

pair_labels = [f"{p['true_class']} → {p['predicted_class']}" for p in top_pairs]
pair_counts = [p['count'] for p in top_pairs]

bars = ax.barh(range(len(top_pairs)), pair_counts, color='coral')
ax.set_yticks(range(len(top_pairs)))
ax.set_yticklabels(pair_labels, fontsize=9)
ax.set_xlabel('Number of Misclassifications', fontsize=11, fontweight='bold')
ax.set_title('Top 15 Most Confused Class Pairs', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()

# Add value labels on bars
for i, (bar, count) in enumerate(zip(bars, pair_counts)):
    ax.text(count + 0.5, i, str(count), va='center', fontsize=9)

plt.tight_layout()
viz_dir = project_root / 'results' / 'visualizations'
viz_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(viz_dir / 'error_analysis_confused_pairs.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Visualization saved")


## 3. Per-Class Error Analysis


In [ ]:
# Calculate per-class error rates
class_error_stats = []
for class_id in range(NUM_CLASSES):
    class_mask = results_df['true_label'] == class_id
    class_samples = results_df[class_mask]
    
    if len(class_samples) > 0:
        error_count = (~class_samples['is_correct']).sum()
        error_rate = error_count / len(class_samples)
        correct_count = class_samples['is_correct'].sum()
        
        # Most common misclassification
        misclassified = class_samples[~class_samples['is_correct']]
        if len(misclassified) > 0:
            most_common_error = misclassified['predicted_class'].value_counts().index[0]
            most_common_error_count = misclassified['predicted_class'].value_counts().iloc[0]
        else:
            most_common_error = "N/A"
            most_common_error_count = 0
        
        class_error_stats.append({
            'class_id': class_id,
            'class_name': id_to_label[class_id],
            'total_samples': len(class_samples),
            'correct': correct_count,
            'errors': error_count,
            'error_rate': error_rate,
            'accuracy': 1 - error_rate,
            'most_common_error': most_common_error,
            'most_common_error_count': most_common_error_count
        })

error_df = pd.DataFrame(class_error_stats)

print("=" * 100)
print("PER-CLASS ERROR ANALYSIS")
print("=" * 100)
print(f"{'Class':<25} {'Samples':<10} {'Correct':<10} {'Errors':<10} {'Error Rate':<12} {'Accuracy':<12} {'Most Common Error':<25}")
print("-" * 100)
for _, row in error_df.iterrows():
    print(f"{row['class_name']:<25} {row['total_samples']:<10} {row['correct']:<10} {row['errors']:<10} "
          f"{row['error_rate']:<12.4f} {row['accuracy']:<12.4f} {row['most_common_error']:<25}")
print("=" * 100)


In [ ]:
# Visualize per-class error rates
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Error rate by class
axes[0, 0].barh(range(len(error_df)), error_df['error_rate'], color='coral')
axes[0, 0].set_yticks(range(len(error_df)))
axes[0, 0].set_yticklabels(error_df['class_name'], fontsize=8)
axes[0, 0].set_xlabel('Error Rate', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Error Rate by Class', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='x')
axes[0, 0].invert_yaxis()

# Accuracy by class
axes[0, 1].barh(range(len(error_df)), error_df['accuracy'], color='lightgreen')
axes[0, 1].set_yticks(range(len(error_df)))
axes[0, 1].set_yticklabels(error_df['class_name'], fontsize=8)
axes[0, 1].set_xlabel('Accuracy', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Accuracy by Class', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='x')
axes[0, 1].set_xlim([0, 1])
axes[0, 1].invert_yaxis()

# Error count by class
axes[1, 0].barh(range(len(error_df)), error_df['errors'], color='salmon')
axes[1, 0].set_yticks(range(len(error_df)))
axes[1, 0].set_yticklabels(error_df['class_name'], fontsize=8)
axes[1, 0].set_xlabel('Number of Errors', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Total Errors by Class', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='x')
axes[1, 0].invert_yaxis()

# Sample count vs error rate
axes[1, 1].scatter(error_df['total_samples'], error_df['error_rate'], 
                   s=100, alpha=0.6, color='steelblue')
axes[1, 1].set_xlabel('Number of Samples', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Error Rate', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Sample Count vs Error Rate', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(viz_dir / 'error_analysis_per_class.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Visualization saved")


## 4. Error Analysis by Text Characteristics


In [ ]:
# Analyze errors by text length
print("=" * 80)
print("ERROR ANALYSIS BY TEXT LENGTH")
print("=" * 80)

# Create length bins
results_df['length_bin'] = pd.cut(
    results_df['text_length'], 
    bins=[0, 200, 500, 1000, 2000, float('inf')],
    labels=['0-200', '200-500', '500-1000', '1000-2000', '2000+']
)

length_analysis = results_df.groupby('length_bin').agg({
    'is_correct': ['count', 'sum', 'mean'],
    'text_length': 'mean'
}).round(2)

length_analysis.columns = ['total_samples', 'correct', 'accuracy', 'avg_length']
length_analysis['errors'] = length_analysis['total_samples'] - length_analysis['correct']
length_analysis['error_rate'] = 1 - length_analysis['accuracy']

print(length_analysis)
print()

# Analyze errors by word count
results_df['word_bin'] = pd.cut(
    results_df['word_count'],
    bins=[0, 50, 100, 200, 500, float('inf')],
    labels=['0-50', '50-100', '100-200', '200-500', '500+']
)

word_analysis = results_df.groupby('word_bin').agg({
    'is_correct': ['count', 'sum', 'mean'],
    'word_count': 'mean'
}).round(2)

word_analysis.columns = ['total_samples', 'correct', 'accuracy', 'avg_words']
word_analysis['errors'] = word_analysis['total_samples'] - word_analysis['correct']
word_analysis['error_rate'] = 1 - word_analysis['accuracy']

print("=" * 80)
print("ERROR ANALYSIS BY WORD COUNT")
print("=" * 80)
print(word_analysis)


In [ ]:
# Visualize error patterns by text characteristics
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Error rate by text length
length_error_rate = results_df.groupby('length_bin')['is_correct'].agg(['count', 'mean'])
length_error_rate['error_rate'] = 1 - length_error_rate['mean']
axes[0, 0].bar(range(len(length_error_rate)), length_error_rate['error_rate'], 
               color='coral', alpha=0.7)
axes[0, 0].set_xticks(range(len(length_error_rate)))
axes[0, 0].set_xticklabels(length_error_rate.index, rotation=45, ha='right')
axes[0, 0].set_ylabel('Error Rate', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Error Rate by Text Length (characters)', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Error rate by word count
word_error_rate = results_df.groupby('word_bin')['is_correct'].agg(['count', 'mean'])
word_error_rate['error_rate'] = 1 - word_error_rate['mean']
axes[0, 1].bar(range(len(word_error_rate)), word_error_rate['error_rate'],
               color='steelblue', alpha=0.7)
axes[0, 1].set_xticks(range(len(word_error_rate)))
axes[0, 1].set_xticklabels(word_error_rate.index, rotation=45, ha='right')
axes[0, 1].set_ylabel('Error Rate', fontsize=11, fontweight='bold')
axes[0, 1].set_title('Error Rate by Word Count', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Confidence distribution: correct vs incorrect
correct_conf = results_df[results_df['is_correct']]['confidence']
incorrect_conf = results_df[~results_df['is_correct']]['confidence']

axes[1, 0].hist(correct_conf, bins=30, alpha=0.6, label='Correct', color='green', density=True)
axes[1, 0].hist(incorrect_conf, bins=30, alpha=0.6, label='Incorrect', color='red', density=True)
axes[1, 0].set_xlabel('Confidence Score', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Density', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Confidence Distribution: Correct vs Incorrect', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Text length distribution: correct vs incorrect
axes[1, 1].hist(results_df[results_df['is_correct']]['text_length'], bins=30, 
                alpha=0.6, label='Correct', color='green', density=True)
axes[1, 1].hist(results_df[~results_df['is_correct']]['text_length'], bins=30,
                alpha=0.6, label='Incorrect', color='red', density=True)
axes[1, 1].set_xlabel('Text Length (characters)', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Density', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Text Length Distribution: Correct vs Incorrect', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(viz_dir / 'error_analysis_text_characteristics.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Visualization saved")


## 5. Confidence Score Analysis


In [ ]:
# Analyze confidence scores for correct vs incorrect predictions
print("=" * 80)
print("CONFIDENCE SCORE ANALYSIS")
print("=" * 80)

correct_conf = results_df[results_df['is_correct']]['confidence']
incorrect_conf = results_df[~results_df['is_correct']]['confidence']

print(f"\nCorrect Predictions:")
print(f"  Mean confidence: {correct_conf.mean():.4f}")
print(f"  Median confidence: {correct_conf.median():.4f}")
print(f"  Min confidence: {correct_conf.min():.4f}")
print(f"  Max confidence: {correct_conf.max():.4f}")
print(f"  Std confidence: {correct_conf.std():.4f}")

print(f"\nIncorrect Predictions:")
print(f"  Mean confidence: {incorrect_conf.mean():.4f}")
print(f"  Median confidence: {incorrect_conf.median():.4f}")
print(f"  Min confidence: {incorrect_conf.min():.4f}")
print(f"  Max confidence: {incorrect_conf.max():.4f}")
print(f"  Std confidence: {incorrect_conf.std():.4f}")

# High confidence errors (false positives with high confidence)
high_conf_errors = results_df[(~results_df['is_correct']) & (results_df['confidence'] > 0.8)]
print(f"\nHigh Confidence Errors (>0.8): {len(high_conf_errors)}")
if len(high_conf_errors) > 0:
    print(f"  These are cases where the model was very confident but wrong")
    print(f"  Mean confidence: {high_conf_errors['confidence'].mean():.4f}")

# Low confidence correct (true positives with low confidence)
low_conf_correct = results_df[(results_df['is_correct']) & (results_df['confidence'] < 0.5)]
print(f"\nLow Confidence Correct (<0.5): {len(low_conf_correct)}")
if len(low_conf_correct) > 0:
    print(f"  These are cases where the model was uncertain but correct")
    print(f"  Mean confidence: {low_conf_correct['confidence'].mean():.4f}")

print("=" * 80)


## 6. Detailed Misclassification Examples


In [ ]:
# Get misclassified samples
misclassified = results_df[~results_df['is_correct']].copy()
misclassified = misclassified.sort_values('confidence', ascending=False)

print("=" * 80)
print("TOP 30 MISCLASSIFIED SAMPLES (by confidence)")
print("=" * 80)

for idx, (_, row) in enumerate(misclassified.head(30).iterrows(), 1):
    print(f"\n{idx}. True: {row['true_class']} → Predicted: {row['predicted_class']}")
    print(f"   Confidence: {row['confidence']:.4f} | Length: {row['text_length']} chars | Words: {row['word_count']}")
    print(f"   Text: {row['text'][:200]}...")
    if idx % 10 == 0:
        print("\n" + "-" * 80)

print("\n" + "=" * 80)


In [ ]:
# Analyze misclassifications by class pairs
print("=" * 80)
print("MISCLASSIFICATIONS BY TRUE CLASS")
print("=" * 80)

for class_id in range(NUM_CLASSES):
    class_errors = misclassified[misclassified['true_label'] == class_id]
    if len(class_errors) > 0:
        print(f"\n{id_to_label[class_id]} ({len(class_errors)} errors):")
        error_dist = class_errors['predicted_class'].value_counts()
        for pred_class, count in error_dist.head(5).items():
            pct = (count / len(class_errors)) * 100
            print(f"  → {pred_class}: {count} ({pct:.1f}%)")

print("\n" + "=" * 80)


## 7. Error Pattern Visualization


In [ ]:
# Create error confusion heatmap (only showing errors, not correct predictions)
error_cm = np.zeros((NUM_CLASSES, NUM_CLASSES))
for _, row in misclassified.iterrows():
    true_id = int(row['true_label'])
    pred_id = int(row['predicted_label'])
    error_cm[true_id, pred_id] += 1

# Normalize by row (true class) to show error distribution
error_cm_normalized = error_cm.copy()
for i in range(NUM_CLASSES):
    row_sum = error_cm[i].sum()
    if row_sum > 0:
        error_cm_normalized[i] = error_cm[i] / row_sum
    else:
        error_cm_normalized[i] = 0

# Plot error confusion matrix
plt.figure(figsize=(16, 14))
sns.heatmap(
    error_cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Reds',
    xticklabels=[id_to_label[i] for i in range(NUM_CLASSES)],
    yticklabels=[id_to_label[i] for i in range(NUM_CLASSES)],
    cbar_kws={'label': 'Proportion of Errors'},
    linewidths=0.5
)
plt.title('Error Confusion Matrix (Normalized by True Class)', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Predicted Class', fontsize=12, fontweight='bold')
plt.ylabel('True Class', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(viz_dir / 'error_analysis_confusion_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Error confusion heatmap saved")


## 8. Summary and Insights


In [ ]:
# Generate summary statistics
print("=" * 80)
print("ERROR ANALYSIS SUMMARY")
print("=" * 80)

total_samples = len(results_df)
total_errors = len(misclassified)
overall_accuracy = results_df['is_correct'].mean()

print(f"\nOverall Statistics:")
print(f"  Total test samples: {total_samples:,}")
print(f"  Correct predictions: {total_samples - total_errors:,} ({overall_accuracy*100:.2f}%)")
print(f"  Misclassifications: {total_errors:,} ({(1-overall_accuracy)*100:.2f}%)")

print(f"\nClass Performance:")
worst_class = error_df.loc[error_df['error_rate'].idxmax()]
best_class = error_df.loc[error_df['error_rate'].idxmin()]
print(f"  Worst performing class: {worst_class['class_name']} (Error rate: {worst_class['error_rate']:.4f})")
print(f"  Best performing class: {best_class['class_name']} (Error rate: {best_class['error_rate']:.4f})")

print(f"\nMost Confused Pairs:")
for pair in confusion_pairs[:5]:
    print(f"  {pair['true_class']} → {pair['predicted_class']}: {pair['count']} errors")

print(f"\nConfidence Analysis:")
print(f"  Mean confidence (correct): {correct_conf.mean():.4f}")
print(f"  Mean confidence (incorrect): {incorrect_conf.mean():.4f}")
print(f"  High confidence errors (>0.8): {len(high_conf_errors)}")

print(f"\nText Characteristics:")
print(f"  Average text length (correct): {results_df[results_df['is_correct']]['text_length'].mean():.0f} chars")
print(f"  Average text length (incorrect): {results_df[~results_df['is_correct']]['text_length'].mean():.0f} chars")

print("\n" + "=" * 80)
print("RECOMMENDATIONS FOR IMPROVEMENT")
print("=" * 80)
print("1. Focus on improving classification for classes with highest error rates")
print("2. Investigate most confused class pairs - may need more training data or feature engineering")
print("3. Consider data augmentation for underrepresented or difficult classes")
print("4. Review high-confidence errors - these may indicate systematic biases")
print("5. Analyze text length patterns - very short or very long texts may need special handling")
print("=" * 80)


In [ ]:
# Save error analysis results
results_dir = project_root / 'results'
results_dir.mkdir(exist_ok=True)

# Save error statistics
error_analysis_summary = {
    'overall_accuracy': float(overall_accuracy),
    'total_samples': int(total_samples),
    'total_errors': int(total_errors),
    'error_rate': float(1 - overall_accuracy),
    'worst_class': {
        'name': worst_class['class_name'],
        'error_rate': float(worst_class['error_rate'])
    },
    'best_class': {
        'name': best_class['class_name'],
        'error_rate': float(best_class['error_rate'])
    },
    'top_confused_pairs': [
        {
            'true_class': p['true_class'],
            'predicted_class': p['predicted_class'],
            'count': p['count']
        }
        for p in confusion_pairs[:10]
    ],
    'confidence_stats': {
        'correct_mean': float(correct_conf.mean()),
        'incorrect_mean': float(incorrect_conf.mean()),
        'high_confidence_errors': int(len(high_conf_errors))
    },
    'per_class_stats': error_df.to_dict('records')
}

error_json_path = results_dir / 'error_analysis_summary.json'
with open(error_json_path, 'w', encoding='utf-8') as f:
    json.dump(error_analysis_summary, f, indent=2, ensure_ascii=False)

print(f"✓ Error analysis summary saved to {error_json_path}")

# Save detailed misclassified samples
misclassified_export = misclassified[['text', 'true_class', 'predicted_class', 'confidence', 
                                     'text_length', 'word_count']].head(100)
misclassified_csv_path = results_dir / 'misclassified_samples.csv'
misclassified_export.to_csv(misclassified_csv_path, index=False, encoding='utf-8')
print(f"✓ Top 100 misclassified samples saved to {misclassified_csv_path}")

print("\n✓ Error analysis complete!")
